#  Non-spherical particles
### Moment Transport vs Hybrid Transport vs Monte Carlo

**Purpose:** Benchmark comparison of three approaches for simulating the evolution of a non-spherical particle population in a fluid:

| Method | Abbreviation | Description |
|---|---|---|
| Hybrid Transport | **HT** | Strang-split moment transport: explicit node advection + CQMOM breakage |
| Moment Transport | **MT** | Direct moment transport: ODE evolution of moments (advection + breakage in one flux) |
| Monte Carlo | **MC** | Stochastic ground-truth particle simulation |

Each particle is described by the state vector $\boldsymbol{\xi} = (d,\, \chi,\, u,\, v,\, w)$, where $d$ is the volume-equivalent diameter, $\chi$ the aspect ratio, and $(u, v, w)$ the velocity components.

## 1. Imports & Global Style

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import itertools
import os
import re
import sys

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.ticker import EngFormatter, LogFormatterMathtext, ScalarFormatter
import numpy as np
from scipy.integrate import solve_ivp

# ── Project modules ───────────────────────────────────────────────────────────
from Config import *
from Tools import ssp_rk_solver
from Tools import CQMOM
from Tools.initial_NDF import generate_copula_mixture_conditions

## 2. Simulation Configuration

In [ ]:
progress_bar_enabled = True

# ── Publication-quality plot style (applied once, reused everywhere) ──────────
PLOT_STYLE = config["plotting"]["style"]
plt.rcParams.update(PLOT_STYLE)

# ── Consistent colour palette across all figures ──────────────────────────────
COLORS = config["plotting"]["colors"]

print("Imports and plot style loaded.")

# ── Create output folder ──────────────────────────────────────────────────────
folder_name = create_simulation_folder(config)
print(f"Output folder : {folder_name}")

# ── Simulation parameters ─────────────────────────────────────────────────────
dim             = config["simulation"]["dim"]
time_step       = config["simulation"]["time_step"]
simulation_time = config["simulation"]["simulation_time"]
num_particles   = config["simulation"]["num_particles"]
num_loops       = int(simulation_time / time_step)
t_span          = (0, time_step * num_loops)
N               = tuple(config["initial_conditions"]["N"])

# ── Physical properties ───────────────────────────────────────────────────────
rho_p = config["physics"]["rho_p"]   # particle density  [kg/m³]
rho_f = config["physics"]["rho_f"]   # fluid density     [kg/m³]
mu_f  = config["physics"]["mu_f"]    # dynamic viscosity [Pa·s]
g     = config["physics"]["g"]       # gravitational acc [m/s²]

# ── Time-stepping parameters ──────────────────────────────────────────────────
dt        = config["simulation"]["time_step"]
min_dt    = config["time_stepping"]["min_dt"]
max_dt    = config["time_stepping"]["max_dt"]
error_tol = config["time_stepping"]["error_tol"]

# ── Marginal distribution types ───────────────────────────────────────────────
DIM_TYPES = config["mixture"]["dim_types"]

# ── Mixture weights ───────────────────────────────────────────────────────────
MIX_WEIGHTS = config["mixture"]["mixture_weights"]

# ── Mode 1 – primary population ───────────────────────────────────────────────
MODE_1 = config["mixture"]["mode_1"]

# ── Mode 2 – secondary population ──────────────────────────────────────────────
MODE_2 = config["mixture"]["mode_2"]

# ── Breakage model constants ──────────────────────────────────────────────────
FRAG_RATE_CONST    = config["breakage"]["frag_rate_const"]
FRAG_POWER         = config["breakage"]["frag_power"]
MIN_FRAG_SIZE      = config["breakage"]["min_frag_size"]
RELAXATION_FACTOR  = config["breakage"]["relaxation_factor"]

# TAU_RELAX controls the timescale of shape relaxation towards sphericity (AR=1).
TAU_RELAX          = config["attrition"]["tau_relax"]
ATTRITION_RATE_CONST = config["attrition"]["attrition_rate_const"]

# ── Fluid forcing constants ───────────────────────────────────────────────
FREQUENCY       = config["fluid_forcing"]["freq"]
U_AMPLITUDE       = config["fluid_forcing"]["u_amp"]
V_AMPLITUDE       = config["fluid_forcing"]["v_amp"]
W_AMPLITUDE       = config["fluid_forcing"]["w_amp"]
OMEGA           = 2.0 * np.pi * FREQUENCY

# ── Velocity restitution & ejection spread ────────────────────────────────────
RESTITUTION_FACTOR = config["restitution"]["restitution_factor"]
SIGMA_KICK         = config["restitution"]["sigma_kick"]

# ── Moore-Penrose inversion tolerance ─────────────────────────────────────────
RCOND_MT = config["cqmom"]["rcond_mt"]

print("Configuration extracted successfully.")

## 3. Initial Conditions

The initial particle population is drawn from a **Gaussian copula mixture**. Each mode is parameterised by its marginal means and standard deviations in physical space; correlations are set to identity (independent marginals) unless overridden.

| Dim | Variable | Marginal |
|-----|----------|----------|
| 0 | Diameter $d$ | Log-normal |
| 1 | Aspect ratio $\chi$ | Beta |
| 2–4 | Velocities $(u,v,w)$ | Normal |

In [ ]:
# ── Generate moments and MC seed particles ────────────────────────────────────
N_moments_shape = tuple(2 * n for n in N)

initial_moments, mc_points, mc_weights = generate_copula_mixture_conditions(mixture_weights=MIX_WEIGHTS, mode_configs=[MODE_1, MODE_2], dim_types=DIM_TYPES, N_moments_shape=N_moments_shape, number_density=num_particles, num_seeds=num_particles)

# ── Moment index arrays ───────────────────────────────────────────────────────
I, J, K, L, M = [np.arange(N_moments_shape[i]) for i in range(5)]

print(f"MC seed particles            : {mc_points.shape[0]:,}")

In [ ]:
# ── Initial distribution – visualisation ─────────────────────────────────────
COL_INFO = [
    {"title": r"Diameter $d$ (mm)",     "unit": "m",   "color": "#E69F00"},
    {"title": r"Aspect ratio $\chi$",   "unit": " ",   "color": "#56B4E9"},
    {"title": r"Velocity $u$ (m/s)",    "unit": "m/s", "color": "#009E73"},
    {"title": r"Velocity $v$ (m/s)",    "unit": "m/s", "color": "#0072B2"},
    {"title": r"Velocity $w$ (m/s)",    "unit": "m/s", "color": "#D55E00"},
]

fig, axs = plt.subplots(1, 5, figsize=(24, 4), sharey=True, constrained_layout=True)

for i, (ax, info) in enumerate(zip(axs, COL_INFO)):
    if i == 0:  # diameter
        data = mc_points[:, i] * 1e3
    else:
        data = mc_points[:, i]
        
    ax.tick_params(axis='both', which='major', labelsize=25)

    # Histogram (density = False → particle counts)
    ax.hist(data, bins=80, density=False, color=info["color"], alpha=0.85, edgecolor="black", linewidth=0.4, histtype="stepfilled", zorder=3)

    # X-axis: engineering format with units
    if i != 1:
        fmt = EngFormatter(useMathText=True)
    else:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))

    yfmt = ScalarFormatter(useMathText=True)
    yfmt.set_scientific(True)
    yfmt.set_powerlimits((-2, 3))
    ax.yaxis.set_major_formatter(yfmt)

    ax.set_xlabel(info["title"])
    if i == 0:
        ax.set_ylabel("Number")

if config["output"]["save_figures"]:
    path = os.path.join(folder_name, "Initial_distributions.pdf")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved → {path}")

plt.show()

## 4. Physics model (shared by all methods)

### 4.1 Drag force
The drag on a non-spherical particle uses a shape-corrected correlation of Ouchene (2020), parameterized by the oblate-spheroid shape factor $K_0(\chi)$.

### 4.2 Breakage and kinetic recoil
A power-law breakage rate $R_{break} = k_b\, d^{\gamma}$ drives binary fragmentation. Each parent produces two daughters of diameter $d' = d / \sqrt[3]{2}$ (volume-conserving split) and aspect ratio $\chi' =  \alpha \chi$. Each daughter gains an opposite velocity recoil and a part of the parent's velocity $\mathbf{u}_{daughter}=e\cdot \mathbf{u}_{parent} \pm \mathbf{v}_{recoil}$.

### 4.3 Surface attrition and spheroidization
A velocity dependant surface attrition $\dot{d}=-k_{shear}||\mathbf{u}_{rel}||^2$, this surface attrition makes the particle more and more spherical $\dot{\chi}=\frac{1-\chi}{\tau_{shape}}$

In [ ]:
def particle_dynamics(t: float, state_flat: np.ndarray) -> np.ndarray:
    """
    Compute the time-derivative of the particle state under drag and gravity.

    The state vector is ordered as (d, χ, u, v, w) repeated over *n_nodes* quadrature
    nodes, flattened to a 1-D array of length ``dim x n_nodes``.

    Parameters
    ----------
    t : float
        Current simulation time [s].
    state_flat : ndarray, shape (dim * n_nodes,)
        Flattened particle state vector.

    Returns
    -------
    ndarray, shape (dim * n_nodes,)
        Flattened time-derivative vector.
    """
    n_nodes = len(state_flat) // dim
    state   = state_flat.reshape(dim, n_nodes)
    d, chi, u, v, w = state
    
    d_safe = np.clip(d, MIN_FRAG_SIZE/FRAG_POWER, 1e12)
    chi_safe = np.clip(chi, 1.0e-6, 1.0 - 1.0e-6)

    # ── Oscillating fluid velocity ────────────────────────────────────────────
    u_f   = U_AMPLITUDE * np.cos(OMEGA * t)
    v_f   = V_AMPLITUDE * np.sin(OMEGA * t)
    w_f   = W_AMPLITUDE * np.cos(OMEGA * t)

    u_rel = u_f - u
    v_rel = v_f - v
    w_rel = w_f - w
    U_mag = np.sqrt(u_rel**2 + v_rel**2 + w_rel**2) 

    Re = rho_f * U_mag * d_safe / mu_f
 
    # ── Oblate-spheroid shape factor K₀(χ) ───────────────────────────────────
    e     = np.sqrt(np.abs(1 - chi_safe**2))
    term1 = 8.0 / 3.0 / (chi_safe ** (1.0 / 3.0))
    with np.errstate(all="ignore"):
        atan_val = np.where(chi_safe < 1.0, np.arctan(e / chi_safe), 0.0)
    atan_val = np.nan_to_num(atan_val)
    term2    = (2 * chi_safe / (1 - chi_safe**2) + 2 * (1 - 2 * chi_safe**2) / ((1 - chi_safe**2)**1.5) * atan_val)
    K0 = np.where(np.abs(1 - chi_safe) < 0.01, 1.0, term1 / term2)

    shape_factor = (K0 + 0.15 * (chi_safe**95.91)  * (Re**0.687) + 0.2927 * ((1 - chi_safe)**0.4374) * (Re**0.7512))

    # ── Drag force ────────────────────────────────────────────────────────────
    Drag_factor = 0.5 * rho_f * (np.pi * d_safe**2 / 4.0) * (24.0 / (Re + 1e-20)) * shape_factor * U_mag

    Fd_x = Drag_factor * u_rel
    Fd_y = Drag_factor * v_rel
    Fd_z = Drag_factor * w_rel

    mass = rho_p * (np.pi / 6.0 * d_safe**3)

    # ── Equations of motion ───────────────────────────────────────────────────
    dudt = Fd_x / mass
    dvdt = Fd_y / mass
    dwdt = Fd_z / mass - g

    # Size and shape (Surface attrition and spherodization)
    dddt = - ATTRITION_RATE_CONST * U_mag**2
    dchidt = (1 - chi_safe) / TAU_RELAX

    return np.vstack([dddt, dchidt, dudt, dvdt, dwdt]).flatten()

def breakage_rate(d: np.ndarray) -> np.ndarray:
    """
    Compute the power-law breakage rate for each particle.

    Parameters
    ----------
    d : ndarray
        Particle diameters [m].

    Returns
    -------
    ndarray
        Breakage rates [s⁻¹]. Zero for particles smaller than MIN_FRAG_SIZE.
    """
    
    rates = FRAG_RATE_CONST * (d**FRAG_POWER)
    rates[d < MIN_FRAG_SIZE] = 0.0
    return rates

def breakage_daughter_props(d_parent: np.ndarray, chi_parent: np.ndarray, u: np.ndarray, v: np.ndarray, w: np.ndarray) -> tuple:
    """
    Compute daughter particle properties after a binary fragmentation event.

    Volume is conserved: each daughter has diameter ``d / 2^(1/3)``.
    The aspect ratio changes according to RELAXATION_FACTOR.
    Velocity is inherited from the parent (restitution is applied separately).

    Parameters
    ----------
    d_parent, chi_parent, u, v, w : ndarray
        Parent particle properties.

    Returns
    -------
    tuple of ndarray
        (d_daughter, chi_daughter, u, v, w) for the daughter particles.
    """
    d_daughter  = d_parent  / (2.0**(1.0 / 3.0))
    chi_daughter = RELAXATION_FACTOR * chi_parent
    return d_daughter, chi_daughter, u, v, w

def moment_kick_expansion(val: np.ndarray, k: int, sigma: float = SIGMA_KICK) -> np.ndarray:
    """
    Compute the k-th raw moment of (val + ξ), where ξ ~ N(0, sigma²).

    Uses the closed-form Gaussian moment expansion:
    E[(val + ξ)^k] = Σ_{j even} C(k,j) · val^(k-j) · σ^j · (j-1)!!

    Parameters
    ----------
    val : ndarray
        Deterministic (mean) velocity values.
    k : int
        Moment order (0-7 handled analytically).
    sigma : float, optional
        Standard deviation of the kick. Default is SIGMA_KICK.

    Returns
    -------
    ndarray
        E[(val + ξ)^k]
    """
    s2 = sigma ** 2
    if k == 0: return np.ones_like(val)
    if k == 1: return val
    if k == 2: return val**2 + s2
    if k == 3: return val**3 + 3 * val * s2
    if k == 4: return val**4 + 6 * val**2 * s2 + 3 * s2**2
    if k == 5: return val**5 + 10 * val**3 * s2 + 15 * val * s2**2
    if k == 6: return val**6 + 15 * val**4 * s2 + 45 * val**2 * s2**2 + 15 * s2**3
    if k == 7: return val**7 + 21 * val**5 * s2 + 105 * val**3 * s2**2 + 105 * val * s2**3
    return val**k   # fallback (no kick correction for k > 7)

print("Physics functions defined.")

## 5. CQMOM Inversion (shared utility)

In [ ]:
def robust_inversion(moments_tensor: np.ndarray, N_target: tuple, conditional_moment_fuction, rcond: float = 1e-30) -> tuple:
    """
    Invert a moments tensor via CQMOM with a two-level fallback strategy.

    **Level 1** - Full CQMOM inversion with the specified regularisation.
    **Level 2** - If CQMOM fails or returns invalid weights, fall back to a
    simple two-node approximation built from the first two moments of each
    dimension independently.

    Parameters
    ----------
    moments_tensor : ndarray
        Moments tensor of shape ``(2*N[0], …, 2*N[dim-1])``.
    N_target : tuple of int
        Number of quadrature nodes per dimension.
    rcond : float, optional
        Regularisation threshold for the internal least-squares solver.

    Returns
    -------
    weights : ndarray, shape (n_nodes,)
    nodes   : ndarray, shape (dim, n_nodes)
    N_used  : tuple of int  - may differ from N_target after fallback
    """
    m0 = moments_tensor[(0, 0, 0, 0, 0)]
    if m0 < 1e-20:
        return np.array([0.0]), np.zeros((dim, 1)), tuple(N_target)

    # ── Level 1: full CQMOM ───────────────────────────────────────────────────
    try:
        slice_shape = tuple(2 * n for n in N_target)
        m_slice = moments_tensor[tuple(slice(0, s) for s in slice_shape)]
        weights, nodes = CQMOM.cqmom(N_target, m_slice, conditional_moment_fuction, adaptive=config["cqmom"]["adaptive"], rmin=config["cqmom"]["rmin"], eabs=config["cqmom"]["eabs"], cutoff=config["cqmom"]["cutoff"], rcond=rcond)
        if np.sum(weights) < 1e-20 or np.any(np.isnan(nodes)) or np.any(weights < 0):
            raise ValueError("CQMOM returned invalid weights or nodes.")
        return weights, nodes, tuple(N_target)

    except Exception:
        pass  # fall through to Level 2

    # ── Level 2: two-node fallback ────────────────────────────────────────────
    print("[warn] CQMOM inversion failed - using two-node fallback.")
    w_fallback = np.array([0.5 * m0, 0.5 * m0])
    n_fallback = np.zeros((dim, 2))

    # Index pairs (μ₁, μ₂) for each dimension
    idx_pairs = [
        ((1,0,0,0,0), (2,0,0,0,0)),
        ((0,1,0,0,0), (0,2,0,0,0)),
        ((0,0,1,0,0), (0,0,2,0,0)),
        ((0,0,0,1,0), (0,0,0,2,0)),
        ((0,0,0,0,1), (0,0,0,0,2)),
    ]
    shape = moments_tensor.shape
    for i, (idx1, idx2) in enumerate(idx_pairs):
        if all(k < s for k, s in zip(idx2, shape)):
            mean = moments_tensor[idx1] / m0
            var  = max(1e-12, moments_tensor[idx2] / m0 - mean**2)
            sig  = np.sqrt(var)
            n_fallback[i, 0] = mean - sig
            n_fallback[i, 1] = mean + sig
        elif all(k < s for k, s in zip(idx1, shape)):
            n_fallback[i, :] = moments_tensor[idx1] / m0

    # Diameters must be positive
    n_fallback[0, :] = np.maximum(n_fallback[0, :], 1e-8)
    return w_fallback, n_fallback, tuple(N_target)

print("CQMOM inversion utility defined.")

def progress_bar(label: str, percent: float, bar_len: int = 50) -> None:
    """Print an in-place ASCII progress bar."""
    filled = int(bar_len * min(1.0, percent))
    bar    = "█" * filled + "░" * (bar_len - filled)
    sys.stdout.write(f"\r{label} {bar}")
    sys.stdout.flush()

## 6. Method 1 - Hybrid Transport (HT)

The HT method uses **Strang splitting** (second-order operator splitting):

1. **Half-step advection** – advance quadrature nodes via `particle_dynamics` for $\Delta t / 2$.
2. **Full-step breakage** – evolve the moments under the breakage source term for $\Delta t$.
3. **Half-step advection** – advance nodes again for $\Delta t / 2$.

Moments are reconstructed from the advected nodes at each sub-step.

In [ ]:
def ht_advect_half_step(moments_tensor: np.ndarray, dt_sub: float, t_current: float) -> np.ndarray:
    """
    Advance moments by advecting the underlying CQMOM quadrature nodes.

    Nodes are integrated forward by ``dt_sub`` with RK23, then moments are
    recomputed from the transported node positions and the original weights.
    The zeroth moment (number density) is conserved by rescaling.

    Parameters
    ----------
    moments_tensor : ndarray
        Current moment tensor.
    dt_sub : float
        Sub-step size [s].
    t_current : float
        Simulation time at the start of the sub-step [s].

    Returns
    -------
    ndarray
        Updated moment tensor after node advection.
    """
    M0_old = moments_tensor[(0, 0, 0, 0, 0)]
    weights, nodes, _ = robust_inversion(moments_tensor, N, CQMOM.compute_conditional_moments_direct)
    if np.sum(weights) < 1e-20:
        return moments_tensor

    # Integrate nodes forward
    try:
        sol = solve_ivp(particle_dynamics, [t_current, t_current + dt_sub], nodes.flatten(), method="RK23", rtol=error_tol)
        transported_nodes = sol.y[:, -1].reshape(dim, -1)
    except Exception:
        transported_nodes = nodes  # keep nodes stationary on solver failure

    # Recompute moments from transported nodes
    new_moments = np.zeros_like(moments_tensor)
    for idx in itertools.product(*[range(s) for s in moments_tensor.shape]):
        k1, k2, k3, k4, k5 = idx
        node_monomials = (transported_nodes[0] ** k1 * transported_nodes[1] ** k2 * transported_nodes[2] ** k3 * transported_nodes[3] ** k4 * transported_nodes[4] ** k5)
        new_moments[idx] = np.dot(weights, node_monomials)

    # Conserve number density
    M0_new = new_moments[(0, 0, 0, 0, 0)]
    if M0_new > 1e-20:
        new_moments *= M0_old / M0_new

    # Enforce minimum variance (prevent degenerate distributions)
    m0    = new_moments[(0, 0, 0, 0, 0)]
    shape = new_moments.shape
    for idx1, idx2 in [
        ((1,0,0,0,0), (2,0,0,0,0)),
        ((0,0,1,0,0), (0,0,2,0,0)),
        ((0,0,0,1,0), (0,0,0,2,0)),
        ((0,0,0,0,1), (0,0,0,0,2)),
    ]:
        if all(k < s for k, s in zip(idx2, shape)):
            m1_sq_over_m0 = new_moments[idx1] ** 2 / m0
            if new_moments[idx2] < m1_sq_over_m0 + 1e-12:
                new_moments[idx2] = m1_sq_over_m0 + 1e-12

    return new_moments

def ht_breakage_flux(t: float, moments_flat: np.ndarray, N_structure: tuple) -> np.ndarray:
    """
    Compute the breakage source term for the moment equations (HT method).

    The source term for moment M_{k₁,…,k₅} is:
        S = Σ_α w_α · b(d_α) · [2 · daughter_monomial(α) − parent_monomial(α)]

    Daughter velocities are expanded with the Gaussian kick distribution.

    Parameters
    ----------
    t : float
        Current time (unused; signature required by ODE solver interface).
    moments_flat : ndarray
        Flattened moment tensor.
    N_structure : tuple
        Quadrature structure (number of nodes per dimension).

    Returns
    -------
    ndarray
        Flattened source tensor dM/dt due to breakage.
    """
    m_shape = tuple(2 * n for n in N_structure)
    m_tensor = moments_flat.reshape(m_shape)
    weights, nodes, _ = robust_inversion(m_tensor, N_structure, CQMOM.compute_conditional_moments_direct)

    if np.sum(weights) < 1e-20:
        return np.zeros_like(moments_flat)

    d, chi, u, v, w = nodes
    rates              = breakage_rate(d)
    d_dau, chi_dau, *_  = breakage_daughter_props(d, chi, u, v, w)

    source = np.zeros(m_shape)
    for idx in itertools.product(*[range(s) for s in m_shape]):
        k1, k2, k3, k4, k5 = idx
        parent_mono = d**k1 * chi**k2 * u**k3 * v**k4 * w**k5
        dau_mono    = (d_dau**k1 * chi_dau**k2 * moment_kick_expansion(RESTITUTION_FACTOR * u, k3) * moment_kick_expansion(RESTITUTION_FACTOR * v, k4) * moment_kick_expansion(RESTITUTION_FACTOR * w, k5))
        source[idx] = np.dot(weights * rates, 2.0 * dau_mono - parent_mono)

    return source.flatten()

print("HT helper functions defined.")

In [ ]:
# ── Run HT simulation ─────────────────────────────────────────────────────────
print("=" * 70)
print("Running METHOD 1 - HYBRID TRANSPORT (HT)")
print("=" * 70)

current_moments_HT = initial_moments.copy()
current_time_HT    = t_span[0]
dt_HT              = dt

HT_results_list = [current_moments_HT.copy()]  # t = 0 snapshot
t_eval_HT       = [current_time_HT]

while current_time_HT < simulation_time:
    # 1 – Half-step advection
    m_half = ht_advect_half_step(current_moments_HT, dt_HT / 2.0, current_time_HT)

    # 2 – Full-step breakage
    m_broken_flat, ts_error = ssp_rk_solver.ssp_rk3(state=m_half.flatten(), time_step=dt_HT, t=current_time_HT + dt_HT / 2.0, momidx=N, compute_flux=ht_breakage_flux, adaptive=True)
    m_broken = m_broken_flat.reshape(initial_moments.shape)

    # 3 – Second half-step advection
    current_moments_HT = ht_advect_half_step(m_broken, dt_HT / 2.0, current_time_HT + dt_HT / 2.0)

    current_time_HT += dt_HT
    dt_HT = ssp_rk_solver.adapt_time_step(dt_HT, ts_error, error_tol, min_dt, max_dt)

    t_eval_HT.append(current_time_HT)
    HT_results_list.append(current_moments_HT.copy())

    if progress_bar_enabled:
        progress_bar("HT", current_time_HT / simulation_time)

HT_Moments = np.stack(HT_results_list, axis=-1)
print(f"\nHT completed.")

## 7. Method 2 – Moment Transport (MT)

In the MT method, advection and breakage are **not split**: the full moment flux

$$\frac{\mathrm{d}M_{k}}{\mathrm{d}t} = \underbrace{\mathcal{F}^{adv}_k}_{\text{advection}} + \underbrace{S_k^{\text{break}}}_{\text{breakage}}$$

is computed simultaneously and integrated with SSP-RK3.

In [ ]:
def mt_monomial_time_derivative(t: float, xi: np.ndarray, k: tuple) -> float:
    """
    Compute d/dt [ξ₁^k₁ · ξ₂^k₂ · ξ₃^k₃ · ξ₄^k₄ · ξ₅^k₅] at a single node.

    Uses the chain rule:  d/dt [ξ^k] = k · ξ^(k-1) · dξ/dt

    Parameters
    ----------
    t : float
        Current time [s].
    xi : ndarray, shape (dim,)
        State at a single quadrature node.
    k : tuple of int
        Multi-index of the monomial.

    Returns
    -------
    float
        Time-derivative of the monomial.
    """
    k1, k2, k3, k4, k5 = k
    d, chi, u, v, w     = xi

    derivs = particle_dynamics(t, xi.flatten()).reshape(dim)
    ddt_d, ddt_chi, ddt_u, ddt_v, ddt_w = derivs

    val = 0.0
    if k1 > 0: val += k1 * d**(k1-1)  * chi**k2  * u**k3  * v**k4  * w**k5  * ddt_d
    if k2 > 0: val += k2 * d**k1  * chi**(k2-1) * u**k3  * v**k4  * w**k5  * ddt_chi
    if k3 > 0: val += k3 * d**k1  * chi**k2  * u**(k3-1) * v**k4  * w**k5  * ddt_u
    if k4 > 0: val += k4 * d**k1  * chi**k2  * u**k3  * v**(k4-1) * w**k5  * ddt_v
    if k5 > 0: val += k5 * d**k1  * chi**k2  * u**k3  * v**k4  * w**(k5-1) * ddt_w
    return val

def mt_breakage_source(nodes: np.ndarray, weights: np.ndarray, N_indices: tuple) -> np.ndarray:
    """
    Compute the breakage source tensor S_k = Birth_k - Death_k.

    Daughter velocity moments are expanded with the Gaussian kick distribution
    (see `moment_kick_expansion`).

    Parameters
    ----------
    nodes : ndarray, shape (dim, n_nodes)
    weights : ndarray, shape (n_nodes,)
    N_indices : tuple

    Returns
    -------
    ndarray
        Flattened breakage source tensor.
    """
    d, chi, u, v, w      = nodes
    rates                = breakage_rate(d)
    d_dau, chi_dau, *_   = breakage_daughter_props(d, chi, u, v, w)

    dims   = tuple(2 * n for n in N_indices)
    source = np.zeros(dims)

    for idx in itertools.product(*[range(s) for s in dims]):
        k1, k2, k3, k4, k5 = idx

        # Death: parent disappears with its exact properties
        death = np.dot(weights * rates, d**k1 * chi**k2 * u**k3 * v**k4 * w**k5)

        # Birth: daughter gets kicked velocities
        birth = np.dot(weights * rates, 2.0 * d_dau**k1 * chi_dau**k2 * moment_kick_expansion(RESTITUTION_FACTOR * u, k3) * moment_kick_expansion(RESTITUTION_FACTOR * v, k4) * moment_kick_expansion(RESTITUTION_FACTOR * w, k5))
        
        source[idx] = birth - death

    return source.flatten()

def mt_full_flux(t: float, moments_flat: np.ndarray, N_struct: tuple) -> np.ndarray:
    """
    Compute the total moment flux dM/dt = advection + breakage for the MT method.

    Parameters
    ----------
    t : float
    moments_flat : ndarray
    N_struct : tuple

    Returns
    -------
    ndarray
        Flattened dM/dt.
    """
    shape        = tuple(2 * n for n in N_struct)
    m_tensor     = moments_flat[: np.prod(shape)].reshape(shape)
    weights, nodes, _ = robust_inversion(m_tensor, N_struct, CQMOM.compute_conditional_moments_pseudo, rcond=RCOND_MT)

    if np.sum(weights) < 1e-20:
        return np.zeros(np.prod(shape))

    # ── Advection contribution ────────────────────────────────────────────────
    flux_adv = np.zeros(shape)
    for idx in itertools.product(*[range(s) for s in shape]):
        flux_adv[idx] = sum(
            weights[n] * mt_monomial_time_derivative(t, nodes[:, n], idx)
            for n in range(nodes.shape[1])
        )

    # ── Breakage contribution ─────────────────────────────────────────────────
    flux_break = mt_breakage_source(nodes, weights, N_struct)

    return flux_adv.flatten() + flux_break

def mt_enforce_constraints(moments_tensor: np.ndarray, N_indices: tuple) -> np.ndarray:
    """
    Enforce the minimum-variance (realizability) constraints on the moments.

    Ensures M_{2k} ≥ M_{k}² / M₀ + ε for all tracked second moments.

    Parameters
    ----------
    moments_tensor : ndarray
    N_indices : tuple

    Returns
    -------
    ndarray
        Corrected moment tensor.
    """
    m0    = moments_tensor[(0, 0, 0, 0, 0)]
    if m0 < 1e-20:
        return moments_tensor

    shape = moments_tensor.shape
    for idx1, idx2 in [
        ((1,0,0,0,0), (2,0,0,0,0)),
        ((0,0,1,0,0), (0,0,2,0,0)),
        ((0,0,0,1,0), (0,0,0,2,0)),
        ((0,0,0,0,1), (0,0,0,0,2)),
    ]:
        if all(k < s for k, s in zip(idx2, shape)):
            lower = max(moments_tensor[idx1]**2 / m0 + 1e-12, moments_tensor[idx1]**2 / m0 * (1.0 + 1e-5))
            if moments_tensor[idx2] < lower:
                moments_tensor[idx2] = lower
    return moments_tensor

print("MT helper functions defined.")

In [ ]:
# ── Run MT simulation ─────────────────────────────────────────────────────────
print("=" * 70)
print("Running METHOD 2 - MOMENT TRANSPORT (MT)")
print("=" * 70)

current_moments_MT = initial_moments.copy()
current_time_MT    = t_span[0]
dt_MT              = dt

MT_results_list = [current_moments_MT.copy()]  # t = 0 snapshot
t_eval_MT       = [current_time_MT]

while current_time_MT < simulation_time:
    updated_flat, ts_error = ssp_rk_solver.ssp_rk3(state=current_moments_MT.flatten(), time_step=dt_MT, t=current_time_MT, momidx=N, compute_flux=mt_full_flux, adaptive=True)
    current_moments_MT = mt_enforce_constraints(updated_flat.reshape(tuple(2 * n for n in N)), N)
    dt_MT = ssp_rk_solver.adapt_time_step(dt_MT, ts_error, error_tol, min_dt, max_dt)

    current_time_MT += dt_MT
    t_eval_MT.append(current_time_MT)
    MT_results_list.append(current_moments_MT.copy())

    if progress_bar_enabled:
        progress_bar("MT", current_time_MT / simulation_time)

MT_Moments = np.stack(MT_results_list, axis=-1)
print(f"\nMT completed.")

## 8. Method 3 – Monte Carlo (MC)

Each particle is a full state vector $(d, \chi, u_p, v_p, w_p)$. At every time step:

1. All particles are advected together with `particle_dynamics` via RK23.
2. Each particle breaks stochastically with probability $R_{break}(d)\, \Delta t$.
3. Two daughters replace the parent; each inherits the parent velocity, possibly
   with an isotropic Gaussian kick $\pm \boldsymbol{\xi} \sim N(0, \sigma_{\text{kick}}^2 I)$.

Moments are computed as weighted sums over the particle population.

In [ ]:
def mc_compute_moments(particles: np.ndarray, I: np.ndarray, J: np.ndarray, K: np.ndarray, L: np.ndarray, M_idx: np.ndarray, w_particle: float) -> np.ndarray:
    """
    Compute a moment snapshot from the current particle ensemble.

    M_{i,j,k,l,m} = w_particle · Σ_α d_α^i · χ_α^j · u_α^k · v_α^l · w_α^m

    Parameters
    ----------
    particles : ndarray, shape (N_part, 5)
        Current particle state matrix.
    I, J, K, L, M_idx : ndarray
        Arrays of moment indices.
    w_particle : float
        Statistical weight per computational particle.

    Returns
    -------
    ndarray
        Moment tensor of shape (len(I), len(J), …).
    """
    p_d, p_ar, p_u, p_v, p_w = particles.T
    snapshot = np.zeros((len(I), len(J), len(K), len(L), len(M_idx)))
    for i in I:
        for j in J:
            for k in K:
                for l in L:
                    for m in M_idx:
                        snapshot[i, j, k, l, m] = (np.sum(p_d**i * p_ar**j * p_u**k * p_v**l * p_w**m) * w_particle)
    return snapshot

print("MC helper function defined.")

In [ ]:
# ── Run MC simulation ─────────────────────────────────────────────────────────
print("=" * 70)
print("Running METHOD 3 - MONTE CARLO (MC)")
print("=" * 70)

current_particles_MC = mc_points.copy()
current_time_MC      = t_span[0]
dt_MC                = dt

# Each computational particle represents w_sim_particle physical particles
w_sim_particle = initial_moments[(0, 0, 0, 0, 0)] / num_particles

# t = 0 snapshot
MC_results_list = [mc_compute_moments(current_particles_MC, I, J, K, L, M, w_sim_particle)]
mc_time_eval    = [current_time_MC]

while current_time_MC < simulation_time:
    n_curr = len(current_particles_MC)

    # 1 – Advect all particles
    sol = solve_ivp(particle_dynamics, [current_time_MC, current_time_MC + dt_MC], current_particles_MC.T.flatten(), method="RK23", rtol=error_tol)
    current_particles_MC = sol.y[:, -1].reshape(dim, n_curr).T

    # 2 – Stochastic breakage
    d_curr   = current_particles_MC[:, 0]
    probs    = breakage_rate(d_curr) * dt_MC
    breaking = np.where(np.random.random(n_curr) < probs)[0]

    if len(breaking) > 0:
        parents = current_particles_MC[breaking]
        d, chi  = parents[:, 0], parents[:, 1]
        u, v, w = parents[:, 2], parents[:, 3], parents[:, 4]
        n_break = len(breaking)

        # Isotropic Gaussian kicks (momentum-conserving: d1 gets +kick, d2 gets -kick)
        kicks = np.random.normal(0.0, SIGMA_KICK, (n_break, 3))

        d_new, chi_new, *_ = breakage_daughter_props(d, chi, u, v, w)

        daughter_1 = np.column_stack([d_new, chi_new, RESTITUTION_FACTOR * u + kicks[:, 0], RESTITUTION_FACTOR * v + kicks[:, 1],RESTITUTION_FACTOR * w + kicks[:, 2]])
        daughter_2 = np.column_stack([d_new, chi_new, RESTITUTION_FACTOR * u - kicks[:, 0], RESTITUTION_FACTOR * v - kicks[:, 1], RESTITUTION_FACTOR * w - kicks[:, 2]])

        # Replace parents with daughter_1; append daughter_2
        current_particles_MC[breaking] = daughter_1
        current_particles_MC = np.vstack([current_particles_MC, daughter_2])

    current_time_MC += dt_MC
    MC_results_list.append(mc_compute_moments(current_particles_MC, I, J, K, L, M, w_sim_particle))
    mc_time_eval.append(current_time_MC)

    if progress_bar_enabled:
        progress_bar("MC", current_time_MC / simulation_time)

MC_Moments = np.stack(MC_results_list, axis=-1)
print(f"\nMC completed.")

## 9. Results

### 9.1 Mean moments comparison

Plots the time evolution of the **mean** of each tracked quantity:
$\langle \xi_i \rangle = M_{\mathbf{e}_i} / M_{\mathbf{0}}$

In [ ]:
# ── Shared plotting utilities ─────────────────────────────────────────────────

def safe_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    """Element-wise division; returns zero where denominator < 1e-20."""
    return np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 1e-20)

def get_moment(M: np.ndarray, idx: list) -> np.ndarray:
    """Extract a 1-D time series from a stacked moment array."""
    return M[idx[0], idx[1], idx[2], idx[3], idx[4]]

def save_figure(fig: plt.Figure, name: str) -> None:
    """Save figure to the simulation output folder if configured."""
    if config["output"].get("save_figures", False):
        path = os.path.join(folder_name, f"{name}.pdf")
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print(f"  Saved → {path}")

# Pre-compute time axes 
mc_time  = np.array(mc_time_eval) 
mt_time  = np.array(t_eval_MT)    
ht_time  = np.array(t_eval_HT)
sim_time = simulation_time        

T_TICKS = [0, sim_time/4, sim_time/2, 3*sim_time/4, sim_time]

FMT_T = EngFormatter(unit="", places=0)   # time axis: engineering notation

print("Plotting utilities ready.")

In [ ]:
# ── Moment comparison (6 panels) ──────────────────────────────────────────────
PLOT_MEAN = True  # True → mean quantities;  False → totals

MOMENT_PANELS = {
    r"Number $N$":          {"indices": [0,0,0,0,0], "unit": "-"},
    r"diameter $d$":        {"indices": [1,0,0,0,0], "unit": "m"},
    r"aspect ratio $\chi$": {"indices": [0,1,0,0,0], "unit": "-"},
    r"velocity $u$":        {"indices": [0,0,1,0,0], "unit": "m/s"},
    r"velocity $v$":        {"indices": [0,0,0,1,0], "unit": "m/s"},
    r"velocity $w$":        {"indices": [0,0,0,0,1], "unit": "m/s"},
}

n_panels = len(MOMENT_PANELS)
ncols    = 3
nrows    = int(np.ceil(n_panels / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 4*nrows),
                         constrained_layout=True, sharex=True)
axes = axes.flatten()

for panel_idx, (desc, cfg) in enumerate(MOMENT_PANELS.items()):
    ax  = axes[panel_idx]
    idx = cfg["indices"]
    is_m0 = (idx == [0,0,0,0,0])

    mc_raw = get_moment(MC_Moments, idx)
    mt_raw = get_moment(MT_Moments, idx)
    ht_raw = get_moment(HT_Moments, idx)

    mc_m0 = get_moment(MC_Moments, [0,0,0,0,0])
    mt_m0 = get_moment(MT_Moments, [0,0,0,0,0])
    ht_m0 = get_moment(HT_Moments, [0,0,0,0,0])

    if PLOT_MEAN and not is_m0:
        mc_plot = safe_divide(mc_raw, mc_m0)
        mt_plot = safe_divide(mt_raw, mt_m0)
        ht_plot = safe_divide(ht_raw, ht_m0)
        prefix  = "Mean"
    else:
        mc_plot, mt_plot, ht_plot = mc_raw, mt_raw, ht_raw
        prefix  = "Total"

    # ── Plot three methods ────────────────────────────────────────────────────
    ax.plot(mc_time, mc_plot, 's-', color=COLORS["mc"], alpha=0.65, markersize=7, markerfacecolor="None", label="Monte Carlo")
    ax.plot(mt_time, mt_plot, '-', color=COLORS["mt"], alpha=0.9, label="Direct moment transport")
    ax.plot(ht_time, ht_plot, '-', color=COLORS["ht"], alpha=0.9, label="Strang-split moment transport")

    
    # ── Axis formatting ───────────────────────────────────────────────────────
    fmt_y = ScalarFormatter(useMathText=True)
    fmt_y.set_scientific(True)
    fmt_y.set_powerlimits((-3, 3))
    ax.yaxis.set_major_formatter(fmt_y)
    ax.set_xticks(T_TICKS)
    ax.set_xlim(0,1)

    # ── Title & labels ────────────────────────────────────────────────────────
    math_sym = re.search(r"\$(.*?)\$", desc)
    sym      = math_sym.group(1) if math_sym else desc
    if is_m0:
        title = r"Number of particles $N_\mathrm{p}$"
    else:
        wrap = lambda s: rf"\langle {s} \rangle" if PLOT_MEAN else rf"\Sigma {s}"
        title = f"{prefix} {desc.split('$')[0].strip()} ${wrap(sym)}$"
    ax.set_title(title, fontsize=30)

    unit = cfg.get("unit", "")
    ax.set_ylabel(f"({unit})" if unit and unit != "-" else "")

    if panel_idx >= n_panels - ncols:
        ax.set_xlabel("Time (s)")
        
# ── Remove empty axes, add shared legend ─────────────────────────────────────
for k in range(n_panels, len(axes)):
    fig.delaxes(axes[k])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.10), frameon=False)

save_figure(fig, "Moments_comparison")
plt.show()

### 9.2 Kinetic energy and granular temperature

**Kinetic energy** (per particle):  $E_{k,x} = \frac{1}{2}\frac{\pi}{6}\rho_p \cdot \langle d^3 u^2 \rangle$

**Granular temperature**: $\Theta_x = \langle u^2 \rangle - \langle u \rangle^2$

In [ ]:
FACTOR_KE = 0.5 * (np.pi / 6.0) * rho_p

PHYSICS_PANELS = {
    r"Kinetic energy $E_{\mathrm{k},x}$":   {"type": "energy",      "indices": [3,0,2,0,0], "scale": FACTOR_KE, "log": False},
    r"Kinetic energy $E_{\mathrm{k},y}$":   {"type": "energy",      "indices": [3,0,0,2,0], "scale": FACTOR_KE, "log": False},
    r"Kinetic energy $E_{\mathrm{k},z}$":   {"type": "energy",      "indices": [3,0,0,0,2], "scale": FACTOR_KE, "log": False},
    r"Granular temperature $\Theta_x$":     {"type": "temperature", "idx_sq": [0,0,2,0,0], "idx_mn": [0,0,1,0,0], "log": True},
    r"Granular temperature $\Theta_y$":     {"type": "temperature", "idx_sq": [0,0,0,2,0], "idx_mn": [0,0,0,1,0], "log": True},
    r"Granular temperature $\Theta_z$":     {"type": "temperature", "idx_sq": [0,0,0,0,2], "idx_mn": [0,0,0,0,1], "log": True},
}

n_panels = len(PHYSICS_PANELS)
fig, axes = plt.subplots(int(np.ceil(n_panels / 3)), 3, figsize=(8*3, 4*int(np.ceil(n_panels/3))), constrained_layout=True)
axes = axes.flatten()

for panel_idx, (title, cfg) in enumerate(PHYSICS_PANELS.items()):
    ax     = axes[panel_idx]
    use_log = cfg["log"]

    mc_m0 = get_moment(MC_Moments, [0,0,0,0,0])
    mt_m0 = get_moment(MT_Moments, [0,0,0,0,0])
    ht_m0 = get_moment(HT_Moments, [0,0,0,0,0])

    if cfg["type"] == "energy":
        scale   = cfg["scale"]
        mc_data = safe_divide(get_moment(MC_Moments, cfg["indices"]) * scale, mc_m0)
        mt_data = safe_divide(get_moment(MT_Moments, cfg["indices"]) * scale, mt_m0)
        ht_data = safe_divide(get_moment(HT_Moments, cfg["indices"]) * scale, ht_m0)
        ylabel  = "(J)"

    else:  # granular temperature  Θ = <u²> - <u>²
        mc_data = (safe_divide(get_moment(MC_Moments, cfg["idx_sq"]), mc_m0) - safe_divide(get_moment(MC_Moments, cfg["idx_mn"]), mc_m0)**2)
        mt_data = (safe_divide(get_moment(MT_Moments, cfg["idx_sq"]), mt_m0) - safe_divide(get_moment(MT_Moments, cfg["idx_mn"]), mt_m0)**2)
        ht_data = (safe_divide(get_moment(HT_Moments, cfg["idx_sq"]), ht_m0) - safe_divide(get_moment(HT_Moments, cfg["idx_mn"]), ht_m0)**2)
        ylabel  = r"(m$^2$/s$^2$)"

    # ── Plot ──────────────────────────────────────────────────────────────────
    plot_fn = ax.semilogy if use_log else ax.plot
    plot_fn(mc_time, mc_data, 's-', color=COLORS["mc"], alpha=0.65, markersize=8, markerfacecolor="None", label="Monte Carlo")
    plot_fn(mt_time, mt_data, '-', color=COLORS["mt"], alpha=0.9, label="Direct moment transport")
    plot_fn(ht_time, ht_data, '-', color=COLORS["ht"], alpha=0.9, label="Strang-split moment transport")

    # ── Axis formatting ───────────────────────────────────────────────────────
    if use_log:
        ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))
    else:
        fmt_y = ScalarFormatter(useMathText=True)
        fmt_y.set_scientific(True)
        fmt_y.set_powerlimits((0, 0))
        ax.yaxis.set_major_formatter(fmt_y)
    
    ax.set_xticks(T_TICKS)
    ax.set_xlim(0,1)
    ax.set_title(title, fontsize=30)
    
    if panel_idx % 3 == 0:
        ax.set_ylabel(ylabel)
    
    if panel_idx < 3:
        ax.set_xticklabels([])
    else:    
        ax.set_xlabel("Time (s)")
        
for k in range(n_panels, len(axes)):
    fig.delaxes(axes[k])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3,
           bbox_to_anchor=(0.5, 1.12), frameon=False)

save_figure(fig, "Physics_comparison")
plt.show()

## 10. Summary

In [ ]:
separator = "=" * 70
print(separator)
print("SIMULATION COMPLETED")
print(separator)
print(f"  {'MC final particles':<30s}: {len(current_particles_MC):>10,}")
print(f"  {'MT time steps':<30s}: {len(t_eval_MT):>10,}")
print(f"  {'HT time steps':<30s}: {len(t_eval_HT):>10,}")
print(separator)
print(f"  Output folder: {folder_name}")
print(separator)